# Fraud Mitigation Agent · Advanced 06 Automated Embeddings Atlas

Extensión opcional: prepara `autoEmbed` y consulta texto. Si la capacidad no está habilitada, se conserva el resultado de la ruta manual.


### Recordemos el agente completo (notebooks 00 → 08)

Lo repetimos aquí para que este notebook corra solo, sin depender de los demás.

In [ ]:
import os
import copy
import uuid
from dataclasses import dataclass, field
from typing import Any, Optional


@dataclass(frozen=True)
class Settings:
    """Configuración central del workshop, cargada desde variables de entorno o Colab Secrets."""
    mongodb_uri: Optional[str] = None
    database_name: str = "fraud_mitigation_agent_workshop"
    source_tag: str = "fraud_mitigation_agent_colab_workshop"
    embedding_dimensions: int = 8

    @classmethod
    def from_env(cls, **overrides):
        values = {
            "mongodb_uri": os.getenv("MONGODB_URI"),
            "database_name": os.getenv("FRAUD_MITIGATION_AGENT_DATABASE", "fraud_mitigation_agent_workshop"),
            "source_tag": os.getenv("FRAUD_MITIGATION_AGENT_SOURCE_TAG", "fraud_mitigation_agent_colab_workshop"),
            "embedding_dimensions": int(os.getenv("FRAUD_MITIGATION_AGENT_EMBEDDING_DIMENSIONS", "8")),
        }
        values.update(overrides)
        return cls(**values)


# --- Base de datos en memoria: imita la interfaz de pymongo (find_one, find,
# replace_one, delete_many, insert_one) para que el workshop corra sin Atlas. ---

def _matches(doc, query):
    for key, expected in query.items():
        actual = doc.get(key)
        if isinstance(expected, dict):
            if "$exists" in expected and (key in doc) != bool(expected["$exists"]):
                return False
            if "$in" in expected and actual not in expected["$in"]:
                return False
        elif actual != expected:
            return False
    return True


def _project(row, projection):
    item = copy.deepcopy(row)
    if not projection:
        return item
    excludes = [key for key, value in projection.items() if value == 0]
    includes = [key for key, value in projection.items() if value == 1]
    if includes:
        return {key: item[key] for key in includes if key in item}
    for key in excludes:
        item.pop(key, None)
    return item


class _InsertResult:
    def __init__(self, inserted_id):
        self.inserted_id = inserted_id


class InMemoryCollection:
    def __init__(self):
        self.rows = []

    def find_one(self, query, projection=None):
        for row in self.rows:
            if _matches(row, query):
                return _project(row, projection)
        return None

    def find(self, query=None, projection=None):
        query = query or {}
        return [_project(row, projection) for row in self.rows if _matches(row, query)]

    def replace_one(self, query, replacement, upsert=False):
        for i, row in enumerate(self.rows):
            if _matches(row, query):
                self.rows[i] = copy.deepcopy(replacement)
                return
        if upsert:
            self.rows.append(copy.deepcopy(replacement))

    def delete_many(self, query):
        self.rows = [row for row in self.rows if not _matches(row, query)]

    def insert_one(self, document):
        item = copy.deepcopy(document)
        item.setdefault("_id", uuid.uuid4().hex)
        self.rows.append(item)
        return _InsertResult(item["_id"])

    def aggregate(self, pipeline):
        # $vectorSearch no está disponible en memoria local a propósito:
        # en el notebook 06 vamos a manejar esto con un fallback de similitud
        # coseno calculado en Python.
        raise RuntimeError("Atlas aggregation unavailable in local memory mode")


class InMemoryDB:
    """Imita `client[database_name]` / `db.coleccion` de pymongo, creando
    colecciones sobre la marcha la primera vez que se acceden."""

    def __init__(self):
        self._collections = {}

    def __getitem__(self, name):
        return self.__getattr__(name)

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        self._collections.setdefault(name, InMemoryCollection())
        return self._collections[name]


def get_client(uri, timeout_ms=10000):
    """Conecta a MongoDB Atlas real. Solo se usa si defines MONGODB_URI."""
    from pymongo import MongoClient
    if not uri:
        raise ValueError("MONGODB_URI is required")
    client = MongoClient(uri, serverSelectionTimeoutMS=timeout_ms)
    client.admin.command("ping")
    return client


def get_database(client, database_name):
    return client[database_name]


import hashlib
import re
import numpy as np


def _token_value(token, dimensions):
    digest = hashlib.sha256(token.encode("utf-8")).digest()
    values = np.frombuffer(digest, dtype=np.uint8)[:dimensions].astype(float)
    return (values / 127.5) - 1.0


def deterministic_embedding(text, dimensions=8):
    """Embedding determinístico para el workshop (hashing, sin modelo ni red).
    Lo usamos desde ya para poder sembrar los datos de ejemplo; en el
    notebook 06 vamos a entender cómo funciona y a construir búsqueda por
    similitud vectorial sobre él."""
    tokens = re.findall(r"[a-zA-Z0-9_áéíóúñ-]+", (text or "").lower())
    if not tokens:
        return [0.0] * dimensions
    vector = np.zeros(dimensions, dtype=float)
    for token in tokens:
        vector += _token_value(token, dimensions)
    norm = np.linalg.norm(vector)
    if norm == 0:
        return [0.0] * dimensions
    return (vector / norm).round(6).tolist()


def _pattern(tx_id, fraud_type, text):
    return {
        "tx_id": tx_id,
        "fraud_confirmed": True,
        "fraud_type": fraud_type,
        "fraud_signature_text": text,
        "embedding": deterministic_embedding(text),
        "source_tag": "fraud_mitigation_agent_synthetic",
    }


def demo_documents():
    # Cuatro "firmas" de fraude conocidas: la colección fraud_patterns que la
    # búsqueda por similitud vectorial va a comparar contra cada transacción nueva.
    patterns = [
        _pattern("pattern-001", "account_takeover", "new device new ip impossible travel odd hour credential reset high amount"),
        _pattern("pattern-002", "card_testing", "many small attempts new ip repeated velocity web checkout"),
        _pattern("pattern-003", "synthetic_identity", "new customer device mismatch unusual geo high amount mobile"),
        _pattern("pattern-004", "money_mule", "rapid transfer beneficiary new device distant geo unusual hour"),
    ]
    # Tres arquetipos de cliente, cada uno emparejado con una transacción que
    # se espera que caiga en una banda de decisión distinta (APPROVE/STEP-UP/DENY).
    customers = [
        {
            "customer_id": "customer-normal",
            "usual_ips": ["198.51.100.10"],
            "usual_devices": ["device-normal-001"],
            "usual_countries": ["MX"],
            "avg_amount": 1200,
            "p95_amount": 4500,
            "transactions_24h": 3,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "customer_id": "customer-stepup",
            "usual_ips": ["198.51.100.20"],
            "usual_devices": ["device-step-001"],
            "usual_countries": ["MX"],
            "avg_amount": 1800,
            "p95_amount": 9000,
            "transactions_24h": 4,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "customer_id": "customer-risky",
            "usual_ips": ["198.51.100.30"],
            "usual_devices": ["device-risky-001"],
            "usual_countries": ["MX"],
            "avg_amount": 2500,
            "p95_amount": 12000,
            "transactions_24h": 2,
            "transactions_10m": 1,
            "source_tag": "fraud_mitigation_agent_synthetic",
        },
    ]
    # tx-normal-001 -> se espera APPROVE, tx-stepup-001 -> se espera STEP-UP,
    # tx-risky-001 -> se espera DENY. Los usamos en todos los notebooks.
    transactions = [
        {
            "tx_id": "tx-normal-001", "customer_id": "customer-normal", "amount": 850,
            "currency": "MXN", "timestamp": "2025-01-15T16:20:00Z", "channel": "web",
            "ip": "198.51.100.10", "device_id": "device-normal-001", "geo_km_from_usual": 2,
            "ground_truth_fraud": False, "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "tx_id": "tx-stepup-001", "customer_id": "customer-stepup", "amount": 12000,
            "currency": "MXN", "timestamp": "2025-01-15T22:40:00Z", "channel": "mobile",
            "ip": "198.51.100.20", "device_id": "device-step-001", "geo_km_from_usual": 120,
            "ground_truth_fraud": False, "source_tag": "fraud_mitigation_agent_synthetic",
        },
        {
            "tx_id": "tx-risky-001", "customer_id": "customer-risky", "amount": 5000000,
            "currency": "COP", "timestamp": "2025-01-15T03:00:00Z", "channel": "mobile",
            "ip": "203.0.113.30", "device_id": "device-new-003", "geo_km_from_usual": 9000,
            "ground_truth_fraud": True, "source_tag": "fraud_mitigation_agent_synthetic",
        },
    ]
    rules = {
        "config_id": "risk_rules_config",
        "version": "demo-v1",
        "enabled": True,
        "thresholds": {
            "high_amount": 1000000,
            "amount_multiplier": 5,
            "impossible_travel_km": 500,
            "velocity_10m": 5,
        },
        "weights": {"vector": 0.40, "signals": 0.35, "rules": 0.25},
        "decision_thresholds": {"approve_max": 39, "step_up_max": 69},
        "source_tag": "fraud_mitigation_agent_synthetic",
    }
    return {"patterns": patterns, "customers": customers, "transactions": transactions, "rules": rules}


def seed_demo_data(db, reset=False):
    """Carga los datos sintéticos en la base (Atlas o InMemoryDB). Es idempotente:
    replace_one(upsert=True) evita duplicados si vuelves a correr esta celda."""
    docs = demo_documents()
    collections = {
        "patterns": db["fraud_patterns"],
        "customers": db["customer_state"],
        "transactions": db["transactions"],
        "rules": db["risk_rules_config"],
    }
    if reset:
        for collection in collections.values():
            collection.delete_many({"source_tag": {"$in": ["fraud_mitigation_agent_synthetic", "fraud_mitigation_agent_colab_workshop"]}})
    for document in docs["patterns"]:
        collections["patterns"].replace_one({"tx_id": document["tx_id"]}, document, upsert=True)
    for document in docs["customers"]:
        collections["customers"].replace_one({"customer_id": document["customer_id"]}, document, upsert=True)
    for document in docs["transactions"]:
        document = dict(document)
        document["fraud_signature_text"] = (
            "new device new ip impossible travel odd hour high amount"
            if document["ground_truth_fraud"] else
            "familiar device familiar ip normal amount"
        )
        document["embedding"] = deterministic_embedding(document["fraud_signature_text"])
        collections["transactions"].replace_one({"tx_id": document["tx_id"]}, document, upsert=True)
    collections["rules"].replace_one({"config_id": docs["rules"]["config_id"]}, docs["rules"], upsert=True)
    return {key: len(value) if isinstance(value, list) else 1 for key, value in docs.items()}


# Los Secrets de Colab no se inyectan solos como variables de entorno: hay
# que leerlos explícitamente con userdata.get(...) y copiarlos a os.environ.
# Cada clave se intenta por separado para que una que no exista (userdata.get
# lanza una excepción, no devuelve None) no tumbe la lectura de las demás.
try:
    from google.colab import userdata
except Exception:
    userdata = None

if userdata is not None:
    for _secret_name in ("MONGODB_URI", "LLM_API_KEY", "LLM_MODEL", "LLM_BASE_URL"):
        try:
            _secret_value = userdata.get(_secret_name)
        except Exception:
            _secret_value = None
        if _secret_value:
            os.environ[_secret_name] = _secret_value

settings = Settings.from_env()
if settings.mongodb_uri:
    # pymongo no viene preinstalado en este notebook (a diferencia de 00, que
    # sí lo instala siempre): solo lo instalamos aquí, justo a tiempo, si de
    # verdad vas a usar Atlas real. El camino en memoria no lo necesita.
    try:
        import pymongo  # noqa: F401
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymongo[srv]"], check=True)
    client = get_client(settings.mongodb_uri)
    db = get_database(client, settings.database_name)
else:
    db = InMemoryDB()
seed_demo_data(db, reset=False)
print("Runtime listo:", type(db).__name__)


class MockLLMProvider:
    """Proveedor offline para Colab: determinístico, sin red y sin API key."""

    def complete(self, prompt, system=None):
        # Coincidencia de palabras clave en vez de un modelo real: alcanza
        # para demostrar la mecánica del agente sin necesitar ninguna API key.
        text = (prompt or "").lower()
        if "fraud mitigation agent" in text or "workshop" in text:
            return "El Fraud Mitigation Agent separa herramientas de contexto, scoring determinístico y decisión auditable."
        if "fraude" in text or "fraud" in text:
            return "El flujo combina reglas, señales de comportamiento y similitud vectorial; el resultado final no depende de una respuesta libre del LLM."
        return "MockLLMProvider: respuesta local reproducible para el workshop."


class OpenAICompatibleProvider:
    """Adaptador opcional: nunca lo exige el camino core. Funciona con la API
    real de OpenAI y con cualquier otro servicio que exponga un endpoint
    compatible con /chat/completions (Groq, NVIDIA NIM, Google AI Studio, un
    servidor local, etc.) — cambiar de proveedor es solo otro
    base_url/api_key/model, sin tocar código. Ver README.md para opciones
    gratuitas."""

    def __init__(self, api_key, model, base_url=None):
        if not api_key:
            raise ValueError("LLM_API_KEY is required for the optional provider")
        if not model:
            raise ValueError("LLM_MODEL is required for the optional provider")
        from openai import OpenAI
        kwargs = {"api_key": api_key}
        if base_url:
            kwargs["base_url"] = base_url
        self.client = OpenAI(**kwargs)
        self.model = model

    def complete(self, prompt, system=None):
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        response = self.client.chat.completions.create(model=self.model, messages=messages, temperature=0)
        return response.choices[0].message.content or ""


@dataclass
class ToolResult:
    """Envoltorio uniforme que devuelve cada herramienta, para que la traza del
    agente sea consistente (nombre, estado, datos, latencia, error)."""
    tool_name: str
    status: str = "success"
    data: Any = None
    evidence: list = field(default_factory=list)
    latency_ms: float = 0.0
    error: Optional[str] = None

    def as_dict(self):
        return {
            "tool_name": self.tool_name,
            "status": self.status,
            "data": self.data,
            "evidence": self.evidence,
            "latency_ms": round(self.latency_ms, 2),
            "error": self.error,
        }


import time


def run_tool(name, fn):
    """Mide el tiempo de `fn()`, atrapa cualquier excepción y siempre
    devuelve un ToolResult. Una herramienta nunca lanza una excepción hacia
    el agente: una falla se convierte en status="error" dentro de la traza."""
    started = time.perf_counter()
    try:
        result = fn()
        if isinstance(result, ToolResult):
            result.latency_ms = (time.perf_counter() - started) * 1000
            return result
        return ToolResult(name, data=result, latency_ms=(time.perf_counter() - started) * 1000)
    except Exception as exc:
        return ToolResult(name, status="error", error=str(exc), latency_ms=(time.perf_counter() - started) * 1000)


def get_transaction(db, transaction_id):
    """Herramienta: obtiene la transacción a evaluar. Primer paso de todo análisis."""
    def work():
        document = db.transactions.find_one({"tx_id": transaction_id}, {"_id": 0})
        if not document:
            raise LookupError(f"Transaction not found: {transaction_id}")
        return document
    return run_tool("get_transaction", work)


def get_customer_state(db, customer_id):
    """Herramienta: obtiene el estado base conocido del cliente (dispositivos,
    IPs, montos habituales). Convierte una transacción aislada en algo que se
    puede comparar contra "lo normal para este cliente"."""
    def work():
        document = db.customer_state.find_one({"customer_id": customer_id}, {"_id": 0})
        if not document:
            raise LookupError(f"Customer state not found: {customer_id}")
        return document
    return run_tool("get_customer_state", work)


def get_rules_config(db):
    return run_tool("get_rules_config", lambda: db.risk_rules_config.find_one({"config_id": "risk_rules_config"}, {"_id": 0}))


def evaluate_rules(db, transaction, customer_state=None):
    """Herramienta: evalúa reglas con umbrales configurables (guardados en
    risk_rules_config, no hard-codeados en el prompt)."""
    def work():
        config = db.risk_rules_config.find_one({"config_id": "risk_rules_config"}, {"_id": 0}) or {}
        thresholds = config.get("thresholds", {})
        state = customer_state or {}
        triggered = []
        amount = float(transaction.get("amount", 0))
        if amount >= thresholds.get("high_amount", float("inf")):
            triggered.append({"code": "high_amount", "severity": "high", "detail": amount})
        # "Nuevo" significa "no está en el set conocido de este cliente": un
        # cliente sin historial (usual_ips/usual_devices vacíos) siempre
        # dispara estas dos reglas, por diseño.
        if transaction.get("ip") not in state.get("usual_ips", []):
            triggered.append({"code": "new_ip", "severity": "medium", "detail": transaction.get("ip")})
        if transaction.get("device_id") not in state.get("usual_devices", []):
            triggered.append({"code": "new_device", "severity": "high", "detail": transaction.get("device_id")})
        if float(transaction.get("geo_km_from_usual", 0)) >= thresholds.get("impossible_travel_km", float("inf")):
            triggered.append({"code": "impossible_travel", "severity": "critical", "detail": transaction.get("geo_km_from_usual")})
        # Extrae la hora directo del timestamp ISO (ej. "...T03:00:00Z" -> 3);
        # a propósito no maneja zonas horarias, para mantenerlo simple.
        hour = int(transaction.get("timestamp", "T12:").split("T")[-1][:2] or 12)
        if hour < 6 or hour >= 23:
            triggered.append({"code": "odd_hour", "severity": "medium", "detail": hour})
        return {"triggered_rules": triggered, "config_version": config.get("version", "unknown"), "weights": config.get("weights", {})}
    return run_tool("evaluate_rules", work)


def analyze_behavior(transaction, customer_state):
    """Herramienta: detecta desviaciones estadísticas respecto a lo normal
    *para este cliente*. A diferencia de las reglas (umbrales fijos para
    todos), estas señales son relativas al historial propio del cliente."""
    def work():
        signals = []
        amount = float(transaction.get("amount", 0))
        average = float(customer_state.get("avg_amount", 0))
        # Guard `average`: un cliente nuevo con avg_amount=0 no tiene
        # baseline del cual desviarse, así que se omite en vez de marcar todo.
        if average and amount > average * 5:
            signals.append({"code": "amount_deviation", "score": 35, "detail": f"{amount} > 5x average {average}"})
        if transaction.get("ip") not in customer_state.get("usual_ips", []):
            signals.append({"code": "ip_deviation", "score": 20, "detail": "IP outside usual set"})
        if transaction.get("device_id") not in customer_state.get("usual_devices", []):
            signals.append({"code": "device_deviation", "score": 25, "detail": "device outside usual set"})
        geo = float(transaction.get("geo_km_from_usual", 0))
        if geo > 500:
            signals.append({"code": "geo_deviation", "score": 25, "detail": f"{geo} km"})
        return {"signals": signals, "baseline": {"avg_amount": average, "p95_amount": customer_state.get("p95_amount")}}
    return run_tool("analyze_behavior", work)


def cosine_similarity(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denominator) if denominator else 0.0


def local_similarity(collection, query_vector, limit=3, source_tag=None):
    """Fallback por fuerza bruta (O(n)): calcula la similitud contra cada
    documento embebido y ordena. Suficiente para el puñado de fraud_patterns
    del workshop; no escalaría a una biblioteca de patrones de producción
    (para eso existe el índice ANN de Atlas Vector Search)."""
    query = {"embedding": {"$exists": True}}
    if source_tag:
        query["source_tag"] = source_tag
    rows = []
    for doc in collection.find(query):
        rows.append({
            "tx_id": doc.get("tx_id"),
            "fraud_confirmed": doc.get("fraud_confirmed"),
            "fraud_type": doc.get("fraud_type"),
            "fraud_signature_text": doc.get("fraud_signature_text"),
            "score": round(cosine_similarity(query_vector, doc["embedding"]), 6),
            "search_mode": "local_fallback",
        })
    return sorted(rows, key=lambda x: x["score"], reverse=True)[:limit]


def search_frauds(collection, query_vector, index_name="fraud_vector_index", path="embedding", limit=3, num_candidates=50, source_tag=None, allow_fallback=True):
    """Intenta un $vectorSearch real de Atlas; si falla (sin Atlas, índice sin
    crear, InMemoryDB, etc.) cae en la similitud coseno local de forma transparente."""
    started = time.perf_counter()
    pipeline = [
        {"$vectorSearch": {
            "index": index_name,
            "path": path,
            "queryVector": query_vector,
            "numCandidates": max(num_candidates, limit),
            "limit": limit,
        }},
        {"$project": {
            "_id": 0,
            "tx_id": 1,
            "fraud_confirmed": 1,
            "fraud_type": 1,
            "fraud_signature_text": 1,
            "score": {"$meta": "vectorSearchScore"},
        }},
    ]
    try:
        rows = list(collection.aggregate(pipeline))
        for row in rows:
            row["search_mode"] = "atlas_vector_search"
        return {
            "results": rows,
            "mode": "atlas_vector_search",
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "error": None,
        }
    except Exception as exc:
        # Except amplio a propósito: InMemoryDB lanza RuntimeError, un índice
        # de Atlas faltante lanza OperationFailure, etc. — todos deben
        # degradar al fallback local en vez de tumbar al agente.
        if not allow_fallback:
            raise
        rows = local_similarity(collection, query_vector, limit, source_tag)
        return {
            "results": rows,
            "mode": "local_fallback",
            "latency_ms": round((time.perf_counter() - started) * 1000, 2),
            "error": str(exc),
        }


def text_for_transaction(transaction, signals=None):
    """Convierte los campos de una transacción en texto corto para embeber,
    cuando no viene un fraud_signature_text precalculado."""
    signals = signals or []
    parts = [
        f"channel {transaction.get('channel', 'unknown')}",
        f"amount {transaction.get('amount', 0)} {transaction.get('currency', 'COP')}",
        f"ip {transaction.get('ip', 'unknown')}",
        f"device {transaction.get('device_id', 'unknown')}",
        f"geo_distance {transaction.get('geo_km_from_usual', 0)} km",
    ]
    parts.extend(signals)
    return " ".join(parts)


def find_similar_fraud(db, transaction, signals=None, index_name="fraud_vector_index", dimensions=8, source_tag=None):
    """Herramienta: compara el embedding de esta transacción contra los
    patrones de fraude conocidos."""
    def work():
        text = transaction.get("fraud_signature_text") or text_for_transaction(transaction, [s.get("code", "") for s in (signals or [])])
        vector = transaction.get("embedding") or deterministic_embedding(text, dimensions)
        result = search_frauds(db.fraud_patterns, vector, index_name=index_name, source_tag=source_tag)
        result["query_text"] = text
        return result
    return run_tool("find_similar_fraud", work)


@dataclass(frozen=True)
class DecisionPolicy:
    """Umbrales que traducen un risk_score de 0-100 a una banda de decisión."""
    approve_max: float = 39
    step_up_max: float = 69
    # cualquier valor por encima de step_up_max es DENY


def _bounded(value):
    return max(0.0, min(100.0, float(value)))


def vector_risk_score(similarity_score):
    # Similitud coseno [-1, 1] -> contribución de riesgo [0, 100].
    return _bounded((float(similarity_score) + 1.0) * 50.0)


def rules_risk_score(triggered_rules):
    # Puntaje fijo por severidad, sumado (no promediado) entre todas las
    # reglas que dispararon: más reglas disparadas significa más riesgo.
    severity = {"low": 15, "medium": 30, "high": 50, "critical": 70}
    return _bounded(sum(severity.get(rule.get("severity", "medium"), 30) for rule in triggered_rules))


def signals_risk_score(signals):
    return _bounded(sum(float(signal.get("score", 0)) for signal in signals))


def score_components(vector_score, signals, triggered_rules, weights=None):
    """Combina las tres fuentes de riesgo en un score ponderado y explicable.
    Cada componente se reporta por separado (no solo el número final) para
    que el evidence document muestre cuánto aportó cada fuente."""
    weights = weights or {"vector": 0.40, "signals": 0.35, "rules": 0.25}
    components = {
        "vector": round(vector_risk_score(vector_score), 4),
        "signals": round(signals_risk_score(signals), 4),
        "rules": round(rules_risk_score(triggered_rules), 4),
    }
    risk_score = sum(components[key] * float(weights.get(key, 0)) for key in components)
    return {
        "risk_score": round(_bounded(risk_score), 2),
        "components": components,
        "weights": weights,
        "formula": "Vector × 0.40 + Señales × 0.35 + Reglas × 0.25",
    }


def decision_from_score(risk_score, policy=None):
    policy = policy or DecisionPolicy()
    score = float(risk_score)
    if score <= policy.approve_max:
        return "APPROVE"
    if score <= policy.step_up_max:
        return "STEP-UP"
    return "DENY"


def build_reason_codes(signals, triggered_rules, similarity_results):
    """Códigos legibles para la evidencia: el por qué, no solo el qué."""
    codes = [x.get("code") for x in signals + triggered_rules if x.get("code")]
    if similarity_results and float(similarity_results[0].get("score", 0)) >= 0.75:
        codes.append("fraud_similarity")
    return list(dict.fromkeys(codes))


from datetime import datetime, timezone


def make_evidence(transaction_id, score_result, decision, signals, rules, similarity, trace=None, config_version="demo-v1", policy_version="risk-v1"):
    """Construye el documento de evidencia: el registro de auditoría con
    suficiente detalle para reconstruir por qué se tomó la decisión, sin
    tener que volver a correr el agente."""
    return {
        "evidence_id": f"evidence-{transaction_id}-{uuid.uuid4().hex[:8]}",
        "transaction_id": transaction_id,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "decision": decision,
        "risk_score": score_result["risk_score"],
        "components": score_result["components"],
        "weights": score_result["weights"],
        "signals": signals,
        "triggered_rules": rules,
        "similarity": similarity,
        # Copia, no referencia: el agente le agrega el resultado de esta
        # misma herramienta a `trace` justo después de llamar a esta función,
        # así que guardar la lista por referencia haría que la evidencia se
        # contenga a sí misma (referencia circular: rompe json.dumps y también
        # la inserción en MongoDB en cuanto se usa persist=True).
        "trace": list(trace) if trace else [],
        "config_version": config_version,
        "policy_version": policy_version,
        "source_tag": "fraud_mitigation_agent_workshop",
    }


def persist_evidence(collection, evidence):
    # Copia para no mutar el dict `evidence` original al insertar.
    document = dict(evidence)
    result = collection.insert_one(document)
    document["_id"] = str(result.inserted_id)
    return document


def score_and_decide(db, transaction, behavior_result, rules_result, similarity_result, trace=None):
    """Herramienta: junta score, decisión, reason codes y evidencia en un solo resultado."""
    def work():
        # Solo la mejor coincidencia (la primera) alimenta el score; la lista
        # completa igual queda en el documento de evidencia.
        similarity = (similarity_result.get("results") or [{}])[0]
        config = db.risk_rules_config.find_one({"config_id": "risk_rules_config"}, {"_id": 0}) or {}
        score_result = score_components(
            similarity.get("score", 0),
            behavior_result.get("signals", []),
            rules_result.get("triggered_rules", []),
            config.get("weights"),
        )
        decision = decision_from_score(score_result["risk_score"])
        reason_codes = build_reason_codes(
            behavior_result.get("signals", []),
            rules_result.get("triggered_rules", []),
            similarity_result.get("results", []),
        )
        evidence = make_evidence(
            transaction["tx_id"], score_result, decision,
            behavior_result.get("signals", []), rules_result.get("triggered_rules", []),
            similarity_result.get("results", []), trace,
            config.get("version", "unknown"), "risk-v1",
        )
        return {"transaction_id": transaction["tx_id"], "decision": decision, "risk_score": score_result["risk_score"], "components": score_result["components"], "reason_codes": reason_codes, "evidence": evidence, "config_version": config.get("version", "unknown")}
    return run_tool("score_and_decide", work)


def persist_decision(db, decision_result):
    """Escribe dos documentos: la evidencia completa (risk_evidence) y un
    registro de outcome compacto (final_outcome), barato de consultar para reportes."""
    def work():
        evidence = persist_evidence(db.risk_evidence, decision_result["evidence"])
        outcome = {
            "transaction_id": decision_result["transaction_id"],
            "decision": decision_result["decision"],
            "risk_score": decision_result["risk_score"],
            "reason_codes": decision_result["reason_codes"],
            "evidence_id": decision_result["evidence"]["evidence_id"],
            "source_tag": "fraud_mitigation_agent_workshop",
        }
        db.final_outcome.replace_one({"transaction_id": outcome["transaction_id"]}, outcome, upsert=True)
        return {"outcome": outcome, "evidence_id": str(evidence.get("_id"))}
    return run_tool("persist_decision", work)


@dataclass
class AgentResponse:
    """Payload final que devuelve FraudAgent, con la traza completa de herramientas."""
    response_type: str
    content: str = ""
    tool_calls: list = field(default_factory=list)
    trace_id: str = ""
    data: dict = field(default_factory=dict)

    def as_dict(self):
        return {
            "type": self.response_type,
            "content": self.content,
            "tool_calls": self.tool_calls,
            "trace_id": self.trace_id,
            "data": self.data,
        }


class FraudAgent:
    """Agente completo: encadena las herramientas en un orden fijo hasta
    llegar a una decisión. A propósito NO es un loop de tool-calling guiado
    por el LLM — la secuencia (transacción -> cliente -> reglas ->
    comportamiento -> similitud -> score/decisión -> persistencia) está fija
    en analyze(). El LLM (self.provider) solo se usa para *explicar* una
    decisión que el código determinístico ya tomó: nunca puede saltarse un
    paso, inventar evidencia ni cambiar el resultado."""

    def __init__(self, db, provider=None):
        self.db = db
        self.provider = provider or MockLLMProvider()

    def answer(self, prompt):
        """Ruta de preguntas libres (sin analizar ninguna transacción)."""
        return self.provider.complete(prompt, system="You are the Fraud Mitigation Agent, a concise workshop assistant.")

    def analyze(self, transaction_id, persist=True):
        """Corre el pipeline completo para una transacción y devuelve la decisión + traza."""
        trace = []
        tx_result = get_transaction(self.db, transaction_id)
        trace.append(tx_result.as_dict())
        if tx_result.status != "success":
            return AgentResponse("error", tx_result.error, trace, str(uuid.uuid4())).as_dict()
        transaction = tx_result.data

        customer_result = get_customer_state(self.db, transaction["customer_id"])
        trace.append(customer_result.as_dict())
        customer = customer_result.data or {}

        rules_result = evaluate_rules(self.db, transaction, customer)
        trace.append(rules_result.as_dict())
        behavior_result = analyze_behavior(transaction, customer)
        trace.append(behavior_result.as_dict())
        similarity_result = find_similar_fraud(self.db, transaction, behavior_result.data.get("signals", []))
        trace.append(similarity_result.as_dict())

        decision_result = score_and_decide(
            self.db, transaction, behavior_result.data,
            rules_result.data, similarity_result.data, trace,
        )
        trace.append(decision_result.as_dict())
        persistence = None
        if persist and decision_result.status == "success":
            persistence = persist_decision(self.db, decision_result.data)
            trace.append(persistence.as_dict())

        data = decision_result.data if decision_result.status == "success" else {}
        data["trace"] = trace
        data["persisted"] = bool(persistence and persistence.status == "success")
        # El LLM solo narra la decisión ya tomada — no puede alterarla,
        # porque se calculó antes de esta llamada.
        content = self.provider.complete(
            f"Explain the fraud decision {data.get('decision')} with score {data.get('risk_score')} and reasons {data.get('reason_codes')}",
            system="Explain only the supplied deterministic evidence.",
        )
        return AgentResponse("final", content, trace, str(uuid.uuid4()), data).as_dict()


### Nueva pieza: embeddings automáticos de Atlas

Hasta acá, `find_similar_fraud` generaba el vector nosotros mismos con `deterministic_embedding` (la ruta manual, obligatoria y gratuita). Atlas también puede generar el embedding del lado del servidor con un modelo real (`autoEmbed`), y aceptar la búsqueda por texto directamente, sin que nosotros calculemos ni enviemos ningún vector. Esta ruta es opcional: requiere una conexión real a Atlas con la capacidad de Automated Embeddings habilitada.

In [ ]:
def auto_embedding_index_definition(path="fraud_signature_text", model="voyage-4"):
    """Definición del índice de Atlas Vector Search con Automated Embeddings."""
    return {
        "fields": [
            {
                "type": "autoEmbed",
                "path": path,
                "model": model,
                "modality": "text",
            }
        ]
    }


def create_auto_embedding_index(collection, name="fraud_auto_embedding_index", path="fraud_signature_text", model="voyage-4"):
    definition = auto_embedding_index_definition(path=path, model=model)
    return collection.create_search_index({
        "name": name,
        "type": "vectorSearch",
        "definition": definition,
    })


def automated_text_search(collection, query_text, index_name="fraud_auto_embedding_index", path="fraud_signature_text", limit=3):
    # query.text en vez de queryVector: Atlas embebe query_text del lado del
    # servidor usando el modelo configurado en el índice, así que nunca
    # tenemos que generar o enviar un vector nosotros mismos.
    pipeline = [
        {"$vectorSearch": {
            "index": index_name,
            "query": {"text": query_text},
            "path": path,
            "limit": limit,
        }},
        {"$project": {
            "_id": 0,
            "tx_id": 1,
            "fraud_confirmed": 1,
            "fraud_type": 1,
            "fraud_signature_text": 1,
            "score": {"$meta": "vectorSearchScore"},
        }},
    ]
    return list(collection.aggregate(pipeline))


### Probemos (requiere `MONGODB_URI` real con Automated Embeddings habilitado)

In [ ]:
if settings.mongodb_uri:
    create_auto_embedding_index(db.fraud_patterns)
    print("Índice autoEmbed creado (o ya existía). Puede tardar unos minutos en quedar READY en Atlas.")
    results = automated_text_search(db.fraud_patterns, "new device new ip impossible travel odd hour high amount")
    print(results)
else:
    print("Esta celda requiere una conexión real a MongoDB Atlas (configura MONGODB_URI). InMemoryDB no soporta autoEmbed; la ruta manual del notebook 06 sigue siendo la base del workshop.")
